In [ ]:
import os
import cv2
from PIL import Image
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Model, save_model, load_model
from tensorflow.keras.optimizers.legacy import Adam, SGD, Adadelta
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
# from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Layer, Conv2D, Activation, GlobalAveragePooling2D, Dense, Input, Dropout
from kerastuner.tuners import RandomSearch, BayesianOptimization
from kerastuner.enginer.hyperparameters import HyperParameters

In [ ]:
import sys
from PIL import Image
sys.modules['Image'] = Image

In [ ]:
from PIL import Image
print(Image.__file__)

In [ ]:
gpu = len(tf.config.list_physical_devices('GPU'))>0
print("GPU is", "available" if gpu else "NOT AVAILABLE")

Load Images and Labels

In [ ]:
train_data_dir = './dataset/train'
val_data_dir = './dataset/validation'
test_data_dir = './dataset/test'
img_width, img_height = 224, 224
batch_size = 12
num_epochs =100

In [ ]:
train_datagen = ImageDataGenerator(
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    preprocessing_function=preprocess_input
)

val_datagen = ImageDataGenerator(
     zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    preprocessing_function=preprocess_input
)

# test_datagen = ImageDataGenerator(
#     zoom_range=0.2,
#     horizontal_flip=True,
#     vertical_flip=True,
#     rotation_range=20,
#     preprocessing_function=preprocess_input
# )

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=132,
    class_mode='binary',
)

validation_generator = val_datagen.flow_from_directory(
    val_data_dir,
    target_size=(img_width, img_height),
    batch_size=48,
    class_mode='binary',
)

# test_generator = test_datagen.flow_from_directory(
#     test_data_dir,
#     target_size=(img_width, img_height),
#     batch_size=491,
#     class_mode='binary',
# )



In [ ]:
train_img, train_labels = train_generator.next()
val_img, val_labels = validation_generator.next()

In [ ]:
def build_model(hp):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Freeze the layers except the base model
    for layer in base_model.layers:
        layer.trainable = True    # Layer freeze or not trainable if the model has the orignal class, but if you are using new class,(you are identifying something new from the image) then you have to train the base model again by selecting trainable = True

    # Add a Global Average Pooling Layer
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Add a fully connected layer with unites determined by hp
    x = Dense(units=hp.Int('Dense_units', min_value=128, max_value=1024, step=128), activation='relu')(x)

    # Add the output layer for binary classification
    predictions = Dense(1, activation='sigmoid')(x)

    model = tf.keras.models.Model(inputs=base_model.input, outputs=predictions)

    # Define optimizer with different learning rates
    
    lr =hp.Choice('learning_rate', values=[1e-5, 1e-6, 
                                           2e-5, 2e-6,
                                           3e-5, 3e-6,])
    optimizer = hp.Choice('optimizer', values=['adam'])

    if optimizer == 'adam':
        optimizer = Adam(learning_rate=lr)
    elif optimizer == 'adadelta':
        optimizer = Adadelta(learning_rate=lr)
    else:
        optimizer = SGD(learning_rate=lr)

    model.compile(optimizer=optimizer, 
                  loss='binary_crossentropy', 
                  metrics=['accuracy']
                  )
    
    return model

In [ ]:
tuner = BayesianOptimization(
    build_model,
    objective='val_loss',
    max_trials=12, # number of hyperparameter combinations to try
    directory='./output',
    project_name='Color Detection'
)

In [ ]:
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

tuner.search(train_img, train_labels,
              epochs=100, 
              batch_size =12, 
              callbacks=[early_stopping],
              verbose=1
              )

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model= tuner.get_best_models(num_models=1)[0]

In [ ]:
test_img, test_labels = test_generator.next()

In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_img, test_labels)
print(f'Test accuracy: {test_accuracy}' )
print(f'Test loss: {test_loss}' )

In [ ]:
y_pred = best_model.predict(test_img)
y_pred_binary = (y_pred > 0.7).astype(int)
true_labels = test_labels

In [ ]:
cm = confusion_matrix(true_labels, y_pred_binary)
print("Confusion Matrix")
print(cm)

In [ ]:
import json

In [ ]:
save_model(best_model, './model/detection_adam_0.0000001.h5')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

# Save the TensorFlow Lite model to a file
with open('./model/model.tflite', 'wb') as f:
  f.write(tflite_model)